# Day 10 — Feature Engineering and Regression

## Goal
Create useful features from existing customer data, build a regression pipeline, and evaluate how accurately it predicts next month's spending.

**Target:** `next_month_spend`

> Complete one task at a time. Write your own code in the empty cells before asking for help.

## Step 1 — Create the dataset
Run this starter code without changing it.

In [93]:
import numpy as np
import pandas as pd

np.random.seed(42)

n = 300

df = pd.DataFrame({
    "age": np.random.randint(18, 70, n),
    "monthly_spend": np.random.randint(20, 200, n),
    "visits_per_month": np.random.randint(1, 20, n),
    "months_as_customer": np.random.randint(1, 60, n),
    "plan_type": np.random.choice(
        ["Basic", "Standard", "Premium"], n
    )
})

plan_bonus = df["plan_type"].map({
    "Basic": 0,
    "Standard": 15,
    "Premium": 30
})

df["next_month_spend"] = (
    10
    + 0.65 * df["monthly_spend"]
    + 2.5 * df["visits_per_month"]
    + 0.3 * df["months_as_customer"]
    + plan_bonus
    + np.random.normal(0, 15, n)
).round(2)

df.head()

,age,monthly_spend,visits_per_month,months_as_customer,plan_type,next_month_spend
0,56,179,17,52,Basic,190.89
1,69,28,4,26,Basic,49.12
2,46,118,12,34,Standard,145.97
3,32,166,9,59,Basic,138.95
4,60,67,19,56,Premium,131.63


## Task 1 — Explore the dataset
Use pandas to answer these questions:

1. How many rows and columns are in the dataset?
2. Are any values missing?
3. What is the average `monthly_spend`?
4. What is the average `next_month_spend` for each `plan_type`?

### Your Task 1 code

In [94]:
df.shape

(300, 6)

In [95]:
df.isnull().sum()

age                   0
monthly_spend         0
visits_per_month      0
months_as_customer    0
plan_type             0
next_month_spend      0
dtype: int64

In [96]:
df["monthly_spend"].mean()


np.float64(111.13666666666667)

In [97]:
df.groupby("plan_type")["next_month_spend"].mean()


plan_type
Basic       117.687745
Premium     150.689000
Standard    133.005612
Name: next_month_spend, dtype: float64

### Task 1 explanation
Explain what your results tell you about the dataset.

**My explanation:**
The data has 300 rows and 6 col. the data set has no missing values. Our target is ("next_month_spend").



## Task 2 — Feature engineering
Create two new columns.

### Feature 1: Spending per visit
Calculate the average amount each customer spends per visit:

`spend_per_visit = monthly_spend / visits_per_month`

### Feature 2: Customer loyalty
Create a column named `loyal_customer`. A customer is loyal if they have been subscribed for at least 24 months.

- `1` = Loyal customer
- `0` = Not a loyal customer

After creating both columns, display the first five rows to verify them.

### Your Task 2 code

Feture 1 

In [98]:
df["spend_per_visit"] = df["monthly_spend"] / df["visits_per_month"]

df["spend_per_visit"].head()

0    10.529412
1     7.000000
2     9.833333
3    18.444444
4     3.526316
Name: spend_per_visit, dtype: float64

Feture 2 

In [99]:
# loyal_customer = []
# if df["months_as_customer"] >= 24: this wouldnt work becuse it was looking at the whole col. 
#     loyal_customer.append(1)
# else:
#     loyal_customer.append(0)      
# df[loyal_customer] = loyal_customer

df["loyal_customer"] = (df["months_as_customer"] >= 24).astype(int)
df["loyal_customer"].head(12)




0     1
1     1
2     1
3     1
4     1
5     1
6     1
7     0
8     1
9     1
10    1
11    1
Name: loyal_customer, dtype: int64

In [100]:
df.head()

,age,monthly_spend,visits_per_month,months_as_customer,plan_type,next_month_spend,spend_per_visit,loyal_customer
0,56,179,17,52,Basic,190.89,10.529412,1
1,69,28,4,26,Basic,49.12,7.000000,1
2,46,118,12,34,Standard,145.97,9.833333,1
3,32,166,9,59,Basic,138.95,18.444444,1
4,60,67,19,56,Premium,131.63,3.526316,1


### Task 2 explanation
In your own words, explain what each new feature represents and why it might help the model.

**My explanation:** The money spent per visit helps us understand how much a customer spends on average each time they visit. The loyal customer feature shows whether someone has been a customer for at least 24 months. This may help the model find out whether long-term customers have different spending habits.

## Task 3 — Prepare the data
1. Create `X` containing the input features. Do not include `next_month_spend`.
2. Create `y` containing the `next_month_spend` target.
3. Identify the numerical and categorical feature columns.
4. Split the data using `test_size=0.2` and `random_state=42`.
5. Confirm that the training set contains 240 customers and the test set contains 60.

### Your Task 3 code

In [101]:
# 1 
X = df.drop("next_month_spend",axis=1 )
# 2
y = df["next_month_spend"]


In [102]:
#3 
cat_fet = X.select_dtypes(include= object).columns.to_list()
num_fet = X.select_dtypes(include="number").columns.to_list()

In [103]:
#4 
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)# tried to use stratify but this is a regression model. 

## Task 4 — Build the regression pipeline
Create a complete pipeline using:

- `StandardScaler` for numerical features
- `OneHotEncoder` for categorical features
- `ColumnTransformer` to combine both types of preprocessing
- `LinearRegression` to predict customer spending

Fit the pipeline using only `X_train` and `y_train`.

**Constraint:** All preprocessing must learn from the training data only.

### Your Task 4 code

In [104]:
from sklearn.preprocessing import StandardScaler , OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

In [105]:
feture_scaling = ColumnTransformer([
    ("numerical",StandardScaler(), num_fet),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_fet)
])

In [106]:
linear_pipe = Pipeline([
    ("scaling", feture_scaling ),
    ("regres" , LinearRegression())
])

In [107]:
linear_pipe.fit(X_train,y_train)

y_pred = linear_pipe.predict(X_test)

## Task 5 — Evaluate the model
Use the trained pipeline to predict `X_test`, then calculate:

- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)
- R-squared (R²)

Display the results with clear labels.

### Your Task 5 code

In [108]:
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score,
    accuracy_score
)

mean = mean_absolute_error(y_test,y_pred)
root = root_mean_squared_error(y_test,y_pred)
r_score = r2_score(y_test, y_pred)

print("root_mean_squared_error: ", mean)
print("root_mean_squared_error: ", root)
print("r2: " , r_score)


root_mean_squared_error:  12.840854591170798
root_mean_squared_error:  15.783114031771838
r2:  0.80439726360317


### Explain your model results
1. What does your MAE mean in dollars? $12.80 this shows how much on avg its off 
2. Why is RMSE usually larger than MAE? Beacuse it penliises for more 
This means the model’s features explain about 80% of the variation in the target
**My explanation:**



## Bonus — Compare models
Train a second Linear Regression model without `spend_per_visit` and `loyal_customer`. Use the same train/test split and compare MAE, RMSE, and R².

**Question:** Did feature engineering improve the predictions? Feature engineering slightly improved the model. The model with the engineered features had a slightly lower MAE and RMSE and a slightly higher R². However, the difference was very small, so the new features did not have a major effect on its predictions.

### Your bonus code

In [109]:
# Remove the engineered features from the original train/test split
X_train_basic = X_train.drop(
    columns=["spend_per_visit", "loyal_customer"]
)

X_test_basic = X_test.drop(
    columns=["spend_per_visit", "loyal_customer"]
)

# Identify numerical and categorical columns
basic_numerical = X_train_basic.select_dtypes(
    include="number"
).columns.to_list()

basic_categorical = X_train_basic.select_dtypes(
    include=object
).columns.to_list()

# Create preprocessing for the basic model
basic_preprocessor = ColumnTransformer([
    ("numerical", StandardScaler(), basic_numerical),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), basic_categorical)
])

# Create the second pipeline
basic_pipeline = Pipeline([
    ("preprocessing", basic_preprocessor),
    ("regression", LinearRegression())
])

# Train and predict
basic_pipeline.fit(X_train_basic, y_train)
basic_predictions = basic_pipeline.predict(X_test_basic)

# Evaluate the second model
basic_mae = mean_absolute_error(y_test, basic_predictions)
basic_rmse = root_mean_squared_error(y_test, basic_predictions)
basic_r2 = r2_score(y_test, basic_predictions)

print("Model Without Engineered Features")
print("MAE:", basic_mae)
print("RMSE:", basic_rmse)
print("R²:", basic_r2)



Model Without Engineered Features
MAE: 12.85392779756265
RMSE: 15.7893188825392
R²: 0.8042434378934799


## Concept questions
Answer these in your own words after completing the code.

### Question 1
What is feature engineering, and why might it improve a machine-learning model?

**My answer:** this is when you use exising fetures to make a new feture. It can help the model see patters 

### Question 2
Why do we use MAE and RMSE instead of precision and recall for this problem?

**My answer:** Because we are using a regression model that predicts specific numerical values. MAE and RMSE measure how far the predicted spending is from the actual spending. Precision and recall are used for classification problems that predict categories.

### Question 3
If the model has an MAE of $12, what does that tell you about its predictions?

**My answer:** That the model is off on average $12 

### Question 4
Can adding more features sometimes make a model perform worse? Why?

**My answer:** Yes,Adding features can make a model perform worse if the features are irrelevant, misleading, or introduce unnecessary noise.



## Day 10 reflection
1. What part was most difficult? Understanding the metrix used for regression. 
2. What mistake did you fix? One mistake I fixed was creating the loyal_customer feature. I originally tried to use an if statement on an entire DataFrame column.
3. What did you learn about feature engineering or regression? I learned that creating new features can help a model, but they do not always make a major improvement.

**My reflection:**

